# Auto Loader Ingestion – Car Workshop Franchise

Incrementally ingests parquet files from Unity Catalog Volumes into Delta tables.

| Widget | Values | Description |
|--------|--------|-------------|
| `TRIGGER_MODE` | `availableNow` / `continuous` | **availableNow** – process all files then stop (use for initial load & Auto Loader testing with single-day data). **continuous** – keep stream running, picks up new files every 30 s. |
| `SINGLE_TABLE` | table name or blank | Leave blank to ingest all 20 tables. Set e.g. `dim_locations` to run one table only. |

**Volume layout:**
```
/Volumes/fake_car_workshop_franchise/
  dim/
    dim_parquet_files/       ← source (generator output)
    autoloader_checkpoints/  ← checkpoints + schema locations (this notebook)
  fact/
    fact_parquet_files/      ← source
    autoloader_checkpoints/  ← checkpoints + schema locations
```

In [ ]:
%sql
-- Checkpoint volumes (one per schema – subfolders per table created automatically)
CREATE VOLUME IF NOT EXISTS fake_car_workshop_franchise.dim.autoloader_checkpoints;
CREATE VOLUME IF NOT EXISTS fake_car_workshop_franchise.fact.autoloader_checkpoints;

In [ ]:
import os
from pyspark.sql.functions import col

print("Imports OK")

In [ ]:
dbutils.widgets.dropdown(
    'TRIGGER_MODE', 'availableNow', ['availableNow', 'continuous'],
    label='Trigger Mode'
)
dbutils.widgets.text(
    'SINGLE_TABLE', '',
    label='Single Table  (blank = all, e.g. dim_locations)'
)

TRIGGER_MODE = dbutils.widgets.get('TRIGGER_MODE')
SINGLE_TABLE = dbutils.widgets.get('SINGLE_TABLE').strip()

print(f'TRIGGER_MODE = {TRIGGER_MODE}')
print(f'SINGLE_TABLE = {SINGLE_TABLE or "(all tables)"}')

In [ ]:
CATALOG              = 'fake_car_workshop_franchise'

DIM_PARQUET_BASE     = f'/Volumes/{CATALOG}/dim/dim_parquet_files'
FACT_PARQUET_BASE    = f'/Volumes/{CATALOG}/fact/fact_parquet_files'
DIM_CHECKPOINT_BASE  = f'/Volumes/{CATALOG}/dim/autoloader_checkpoints'
FACT_CHECKPOINT_BASE = f'/Volumes/{CATALOG}/fact/autoloader_checkpoints'

print(f'Source  DIM :  {DIM_PARQUET_BASE}')
print(f'Source  FACT:  {FACT_PARQUET_BASE}')
print(f'Chkpts  DIM :  {DIM_CHECKPOINT_BASE}')
print(f'Chkpts  FACT:  {FACT_CHECKPOINT_BASE}')

In [ ]:
def schema_to_ddl(d):
    return ', '.join(f'`{c}` {t}' for c, t in d.items())


TABLE_SCHEMAS = {
    # ── dimensions ──────────────────────────────────────────────────────
    'dim_locations': {
        'location_id': 'BIGINT', 'location_code': 'STRING', 'name': 'STRING',
        'type': 'STRING', 'street': 'STRING', 'city': 'STRING',
        'region': 'STRING', 'postal_code': 'STRING', 'latitude': 'DOUBLE',
        'longitude': 'DOUBLE', 'phone': 'STRING', 'email': 'STRING',
        'manager_id': 'BIGINT', 'number_of_bays': 'BIGINT', 'area_m2': 'BIGINT',
        'opening_date': 'DATE', 'is_active': 'BOOLEAN',
    },
    'dim_employees': {
        'employee_id': 'BIGINT', 'employee_code': 'STRING', 'first_name': 'STRING',
        'last_name': 'STRING', 'national_id': 'STRING', 'position': 'STRING',
        'location_id': 'BIGINT', 'hire_date': 'DATE', 'termination_date': 'DATE',
        'hourly_rate': 'DOUBLE', 'is_active': 'BOOLEAN',
    },
    'dim_customers': {
        'customer_id': 'BIGINT', 'customer_code': 'STRING', 'customer_type': 'STRING',
        'first_name': 'STRING', 'last_name': 'STRING', 'company_name': 'STRING',
        'tax_id': 'STRING', 'email': 'STRING', 'phone': 'STRING',
        'city': 'STRING', 'postal_code': 'STRING', 'registration_date': 'DATE',
        'preferred_location_id': 'BIGINT', 'marketing_consent': 'BOOLEAN',
    },
    'dim_vehicles': {
        'vehicle_id': 'BIGINT', 'customer_id': 'BIGINT', 'make': 'STRING',
        'model': 'STRING', 'year': 'BIGINT', 'vin': 'STRING',
        'registration_number': 'STRING', 'fuel_type': 'STRING',
        'engine_displacement': 'DOUBLE', 'horsepower': 'BIGINT',
        'color': 'STRING', 'mileage_km': 'BIGINT', 'first_registration_date': 'DATE',
    },
    'dim_products': {
        'product_id': 'BIGINT', 'product_code': 'STRING', 'name': 'STRING',
        'category': 'STRING', 'manufacturer': 'STRING',
        'purchase_price_net': 'DOUBLE', 'sale_price_net': 'DOUBLE',
        'vat_rate': 'BIGINT', 'unit': 'STRING', 'weight_kg': 'DOUBLE',
        'min_stock_level': 'BIGINT', 'is_active': 'BOOLEAN',
    },
    'dim_services': {
        'service_id': 'BIGINT', 'service_code': 'STRING', 'name': 'STRING',
        'category': 'STRING', 'min_price_net': 'BIGINT', 'max_price_net': 'BIGINT',
        'estimated_time_min': 'BIGINT', 'is_active': 'BOOLEAN',
    },
    'dim_suppliers': {
        'supplier_id': 'BIGINT', 'supplier_code': 'STRING', 'name': 'STRING',
        'tax_id': 'STRING', 'city': 'STRING', 'address': 'STRING',
        'postal_code': 'STRING', 'phone': 'STRING', 'email': 'STRING',
        'contact_person': 'STRING', 'payment_terms_days': 'BIGINT',
        'min_order_value': 'DOUBLE', 'is_active': 'BOOLEAN',
    },
    # ── facts ────────────────────────────────────────────────────────────
    'fact_work_orders': {
        'work_order_id': 'BIGINT', 'work_order_code': 'STRING',
        'location_id': 'BIGINT', 'customer_id': 'BIGINT', 'vehicle_id': 'BIGINT',
        'mechanic_id': 'BIGINT', 'reception_date': 'DATE', 'completion_date': 'DATE',
        'status': 'STRING', 'mileage_at_reception': 'BIGINT',
        'customer_notes': 'STRING', 'year': 'BIGINT', 'month': 'BIGINT',
    },
    'fact_work_order_items': {
        'wo_item_id': 'BIGINT', 'work_order_id': 'BIGINT', 'item_type': 'STRING',
        'service_id': 'BIGINT', 'product_id': 'BIGINT', 'quantity': 'BIGINT',
        'unit_price_net': 'DOUBLE', 'value_net': 'DOUBLE', 'vat_rate': 'BIGINT',
        'value_gross': 'DOUBLE', 'discount_percent': 'BIGINT',
    },
    'fact_sales_transactions': {
        'transaction_id': 'BIGINT', 'transaction_code': 'STRING',
        'location_id': 'BIGINT', 'customer_id': 'BIGINT', 'employee_id': 'BIGINT',
        'transaction_date': 'TIMESTAMP', 'payment_method': 'STRING',
        'receipt_number': 'STRING', 'year': 'BIGINT', 'month': 'BIGINT',
    },
    'fact_sales_items': {
        'sales_item_id': 'BIGINT', 'transaction_id': 'BIGINT',
        'product_id': 'BIGINT', 'quantity': 'BIGINT', 'unit_price_net': 'DOUBLE',
        'discount_percent': 'BIGINT', 'value_net': 'DOUBLE',
        'vat_rate': 'BIGINT', 'value_gross': 'DOUBLE',
    },
    'fact_invoices': {
        'invoice_id': 'BIGINT', 'invoice_code': 'STRING',
        'document_type': 'STRING', 'source_type': 'STRING', 'source_id': 'BIGINT',
        'customer_id': 'BIGINT', 'location_id': 'BIGINT',
        'issue_date': 'DATE', 'sale_date': 'DATE', 'payment_due_date': 'DATE',
        'value_net': 'DOUBLE', 'value_vat': 'DOUBLE', 'value_gross': 'DOUBLE',
        'status': 'STRING', 'year': 'BIGINT', 'month': 'BIGINT',
    },
    'fact_payments': {
        'payment_id': 'BIGINT', 'invoice_id': 'BIGINT', 'payment_date': 'DATE',
        'amount': 'DOUBLE', 'payment_method': 'STRING', 'status': 'STRING',
        'transaction_number': 'STRING', 'year': 'BIGINT', 'month': 'BIGINT',
    },
    'fact_inventory_movements': {
        'movement_id': 'BIGINT', 'product_id': 'BIGINT', 'location_id': 'BIGINT',
        'movement_type': 'STRING', 'quantity': 'BIGINT', 'movement_date': 'DATE',
        'source_document': 'STRING', 'document_number': 'STRING',
        'value_net': 'DOUBLE', 'notes': 'STRING', 'year': 'BIGINT', 'month': 'BIGINT',
    },
    'fact_appointments': {
        'appointment_id': 'BIGINT', 'customer_id': 'BIGINT', 'vehicle_id': 'BIGINT',
        'location_id': 'BIGINT', 'service_id': 'BIGINT', 'booking_date': 'DATE',
        'appointment_date': 'TIMESTAMP', 'status': 'STRING',
        'booking_channel': 'STRING', 'notes': 'STRING',
        'year': 'BIGINT', 'month': 'BIGINT',
    },
    'fact_purchase_orders': {
        'po_id': 'BIGINT', 'po_code': 'STRING', 'supplier_id': 'BIGINT',
        'location_id': 'BIGINT', 'order_date': 'DATE',
        'planned_delivery_date': 'DATE', 'actual_delivery_date': 'DATE',
        'value_net': 'DOUBLE', 'value_gross': 'DOUBLE',
        'status': 'STRING', 'year': 'BIGINT',
    },
    'fact_purchase_order_items': {
        'po_item_id': 'BIGINT', 'po_id': 'BIGINT', 'product_id': 'BIGINT',
        'quantity_ordered': 'BIGINT', 'quantity_delivered': 'BIGINT',
        'unit_price_net': 'DOUBLE', 'value_net': 'DOUBLE',
    },
    'fact_customer_feedback': {
        'feedback_id': 'BIGINT', 'customer_id': 'BIGINT', 'location_id': 'BIGINT',
        'work_order_id': 'BIGINT', 'feedback_date': 'DATE', 'rating': 'BIGINT',
        'comment': 'STRING', 'category': 'STRING', 'channel': 'STRING',
    },
    'fact_loyalty_program': {
        'loyalty_id': 'BIGINT', 'customer_id': 'BIGINT', 'event_date': 'DATE',
        'event_type': 'STRING', 'points': 'BIGINT', 'description': 'STRING',
        'balance_after': 'BIGINT', 'tier': 'STRING',
    },
    'fact_employee_schedules': {
        'schedule_id': 'BIGINT', 'employee_id': 'BIGINT', 'date': 'DATE',
        'start_hour': 'BIGINT', 'end_hour': 'BIGINT', 'shift_type': 'STRING',
        'overtime_hours': 'BIGINT', 'attendance': 'STRING',
    },
}

# fact tables that carry partition columns in their parquet directory structure
PARTITIONED_TABLES = {
    'fact_work_orders':         ['year', 'month'],
    'fact_sales_transactions':  ['year', 'month'],
    'fact_invoices':            ['year', 'month'],
    'fact_payments':            ['year', 'month'],
    'fact_inventory_movements': ['year', 'month'],
    'fact_appointments':        ['year', 'month'],
    'fact_purchase_orders':     ['year'],
}

print(f'Schemas loaded: {len(TABLE_SCHEMAS)} tables')

In [ ]:
# Keeps references to running streams when TRIGGER_MODE = continuous
active_streams = []


def ingest_table(table_name, schema_name, trigger_mode='availableNow'):
    is_dim          = schema_name == 'dim'
    source_base     = DIM_PARQUET_BASE     if is_dim else FACT_PARQUET_BASE
    checkpoint_base = DIM_CHECKPOINT_BASE  if is_dim else FACT_CHECKPOINT_BASE

    source_path     = f'{source_base}/{table_name}'
    checkpoint_path = f'{checkpoint_base}/{table_name}/checkpoint'
    schema_location = f'{checkpoint_base}/{table_name}/schema'
    target_table    = f'{CATALOG}.{schema_name}.{table_name}'
    partition_cols  = PARTITIONED_TABLES.get(table_name)

    print(f'  {table_name}  ->  {target_table}')

    reader = (
        spark.readStream
            .format('cloudFiles')
            .option('cloudFiles.format', 'parquet')
            .option('cloudFiles.schemaLocation', schema_location)
            .option('cloudFiles.inferColumnTypes', 'false')   # use our explicit schema
            .schema(schema_to_ddl(TABLE_SCHEMAS[table_name]))
            .load(source_path)
    )

    writer = (
        reader.writeStream
            .format('delta')
            .outputMode('append')
            .option('checkpointLocation', checkpoint_path)
            .option('mergeSchema', 'true')
    )

    if partition_cols:
        writer = writer.partitionBy(*partition_cols)

    if trigger_mode == 'availableNow':
        # Process all currently available files, then stop automatically
        query = writer.trigger(availableNow=True).toTable(target_table)
        query.awaitTermination()
        print(f'     done  ({query.lastProgress["numInputRows"] if query.lastProgress else "?"} rows in last micro-batch)')
    else:
        # Keep stream alive; picks up new files every 30 s
        query = writer.trigger(processingTime='30 seconds').toTable(target_table)
        active_streams.append((table_name, query))
        print(f'     stream started  (id={query.id})')

    return query


print('ingest_table() ready')

## Dimension Tables

7 tables – no partitioning.

In [ ]:
DIM_TABLES = [
    'dim_locations',
    'dim_employees',
    'dim_customers',
    'dim_vehicles',
    'dim_products',
    'dim_services',
    'dim_suppliers',
]

print('=== Dimension tables ===')
for table in DIM_TABLES:
    if not SINGLE_TABLE or SINGLE_TABLE == table:
        ingest_table(table, 'dim', TRIGGER_MODE)
print('Done.')

## Fact Tables

13 tables – 7 are partitioned by `year` / `month` in the parquet layout.

In [ ]:
FACT_TABLES = [
    'fact_work_orders',           # partitioned year/month
    'fact_work_order_items',
    'fact_sales_transactions',    # partitioned year/month
    'fact_sales_items',
    'fact_invoices',              # partitioned year/month
    'fact_payments',              # partitioned year/month
    'fact_inventory_movements',   # partitioned year/month
    'fact_appointments',          # partitioned year/month
    'fact_purchase_orders',       # partitioned year
    'fact_purchase_order_items',
    'fact_customer_feedback',
    'fact_loyalty_program',
    'fact_employee_schedules',
]

print('=== Fact tables ===')
for table in FACT_TABLES:
    if not SINGLE_TABLE or SINGLE_TABLE == table:
        ingest_table(table, 'fact', TRIGGER_MODE)
print('Done.')

## Stop All Continuous Streams

Run the cell below to gracefully stop every active stream  
(only relevant when `TRIGGER_MODE = continuous`).

In [ ]:
if not active_streams:
    print('No active streams to stop.')
else:
    for table_name, query in active_streams:
        query.stop()
        print(f'Stopped: {table_name}')
    active_streams.clear()
    print('All streams stopped.')

## Validation

Row counts for every ingested table.

In [ ]:
all_tables = [('dim', t) for t in DIM_TABLES] + [('fact', t) for t in FACT_TABLES]

results = []
for schema_name, table_name in all_tables:
    full_name = f'{CATALOG}.{schema_name}.{table_name}'
    try:
        count = spark.table(full_name).count()
        results.append({'table': full_name, 'row_count': count, 'status': 'OK'})
    except Exception as e:
        results.append({'table': full_name, 'row_count': None, 'status': str(e)[:60]})

import pandas as pd
df_results = pd.DataFrame(results)
print(df_results.to_string(index=False))